In [2]:
%pip install pandas -q
%pip install torch torchvision -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
import torch
import torch.nn as nn
import torchvision

class CRNN(nn.Module):
    def __init__(self, num_classes: int = 11, hidden_size: int = 128):
        super().__init__()

        # self.cnn = nn.Sequential(
        #     nn.Conv2d(3, 32, kernel_size=3, padding=1),
        #     nn.BatchNorm2d(32),
        #     nn.ReLU(),
        #     nn.MaxPool2d((2, 2)),

        #     nn.Conv2d(32, 64, kernel_size=3, padding=1),
        #     nn.BatchNorm2d(64),
        #     nn.ReLU(),
        #     nn.MaxPool2d((2, 2)),

        #     nn.Conv2d(64, 128, kernel_size=3, padding=1),
        #     nn.BatchNorm2d(128),
        #     nn.ReLU(),
        #     nn.MaxPool2d((2, 1)),

        #     nn.Conv2d(128, 256, kernel_size=3, padding=1),
        #     nn.BatchNorm2d(256),
        #     nn.ReLU(),
        #     nn.MaxPool2d((2, 1)),

        #     nn.Conv2d(256, 256, kernel_size=3, padding=1),
        #     nn.BatchNorm2d(256),
        #     nn.ReLU(),
        #     nn.MaxPool2d((8, 1)),
        # )

        self.backbone = torchvision.models.mobilenet_v3_small(weights="IMAGENET1K_V1").features[:-4]

        # for param in self.backbone.parameters():
        #     param.requires_grad = False

        # for name, child in self.backbone.named_children():
        #     if isinstance(child, nn.Conv2d):
        #         child.padding_mode = 'replicate'
        
        self.neck = nn.Sequential(
            nn.Conv2d(48, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, None)),
        )

        self.dropout = nn.Dropout(0.3)

        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=hidden_size,
            num_layers=2,
            bidirectional=True,
            batch_first=False,
            dropout=0.3
        )

        self.linear = nn.Linear(hidden_size * 2, num_classes)


    def forward(self, x):
        # x = self.cnn(x)
        x = self.backbone(x)
        x = self.neck(x)
        x = x.squeeze(2)
        x = x.permute(2, 0, 1)
        x = self.dropout(x)
        x, _ = self.lstm(x)
        x = self.linear(x)
        return x

In [ ]:
import pandas as pd
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image
import torchvision.transforms.functional as F
import torch.nn.functional as nnF

class DigitDataset(Dataset):
    def __init__(self, csv_path: Path, target_height: int = 128, skip_empty: bool = False, transform=None) -> None:
        self.csv_dir = Path(csv_path).parent
        self.data = pd.read_csv(csv_path, header=None, names=["image", "label"], dtype={"label": str})
        if skip_empty:
            self.data = self.data[self.data["label"].notna() & (self.data["label"].str.strip() != "")]
        self.target_height = target_height
        self.transform = transform


    def __len__(self):
        return len(self.data)
    

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        image = Image.open(self.csv_dir / row["image"]).convert("RGB")

        w, h = image.size
        new_w = int(w * self.target_height / h)
        image = image.resize((new_w, self.target_height))

        if self.transform is not None:
            image = self.transform(image)
        else:
            image = F.to_tensor(image)
            image = F.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        label = [int(c) for c in row["label"]] if pd.notna(row["label"]) and str(row["label"]) != "" else []

        return image, label, new_w
    

def collate_fn(batch):
    images, labels, widths = zip(*batch)

    max_width = max(img.shape[2] for img in images)
    padded = [nnF.pad(img, (0, max_width - img.shape[2])) for img in images]

    return torch.stack(padded), list(labels), list(widths)

In [27]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path

ctc_loss = nn.CTCLoss(blank=10, reduction='mean', zero_infinity=True)

def train(model: CRNN, train_dataset: DigitDataset, val_dataset: DigitDataset, epochs: int = 50, lr: float = 1e-3, batch_size: int = 16):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    # Use different learning rates for backbone and neck/head
    optimizer = torch.optim.Adam([
        {"params": model.backbone.parameters(), "lr": lr},
        {"params": model.neck.parameters(), "lr": lr},
        {"params": model.lstm.parameters(), "lr": lr},
        {"params": model.linear.parameters(), "lr": lr}
    ])
    # optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, labels, widths in train_loader:
            images = images.to(device)

            optimizer.zero_grad()

            logits = model(images)
            log_probs = torch.log_softmax(logits, dim=2)

            T = logits.shape[0]
            input_lengths = torch.clamp(torch.tensor([(w // 16) + 1 for w in widths], dtype=torch.long), max=T)

            target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
            targets = torch.tensor([d for l in labels for d in l], dtype=torch.long)

            loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()
            train_loss += loss.item()
    
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, labels, widths in val_loader:
                images = images.to(device)
                logits = model(images)
                log_probs = torch.log_softmax(logits, dim=2)
                T = logits.shape[0]
                input_lengths = torch.clamp(torch.tensor([(w // 16) + 1 for w in widths], dtype=torch.long), max=T)
                target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
                targets = torch.tensor([d for l in labels for d in l], dtype=torch.long)
                val_loss += ctc_loss(log_probs, targets, input_lengths, target_lengths).item()
            logits = model(images[0].unsqueeze(0).to(device))
            torch.set_printoptions(sci_mode=False)
            probs = torch.softmax(logits[:, 0, :], dim=1)
            print(probs)
            blank_prob = probs[:, 10].mean().item()
            print(f"blank prob: {blank_prob:.3f}")
        
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1}/{epochs} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")
    
    return model

In [28]:
import torchvision.transforms as transforms

model = CRNN()
data_dir = Path("data/digits")
train_transform = transforms.Compose([
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3),
    ], p=0.3),
    
    transforms.RandomAffine(degrees=5, translate=(0.02, 0.02), scale=(0.9, 1.1)),
    
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_dataset = DigitDataset(data_dir / "train.csv", skip_empty=True, transform=train_transform)
val_dataset = DigitDataset(data_dir / "val.csv", skip_empty=True)

dummy = torch.randn(1, 3, 128, 128)
with torch.no_grad():
    out = model(dummy)
print(out.shape)  # (batch, channels, H, W)

model = train(model, train_dataset, val_dataset)

torch.Size([8, 1, 11])
tensor([[0.0283, 0.1605, 0.0225, 0.0417, 0.0732, 0.0201, 0.0645, 0.0147, 0.0244,
         0.0325, 0.5177],
        [0.0184, 0.1468, 0.0141, 0.0320, 0.0663, 0.0114, 0.0568, 0.0084, 0.0164,
         0.0214, 0.6080],
        [0.0149, 0.1404, 0.0114, 0.0285, 0.0638, 0.0089, 0.0543, 0.0064, 0.0136,
         0.0176, 0.6402],
        [0.0137, 0.1378, 0.0105, 0.0273, 0.0632, 0.0080, 0.0536, 0.0058, 0.0126,
         0.0162, 0.6514],
        [0.0132, 0.1364, 0.0101, 0.0268, 0.0629, 0.0077, 0.0536, 0.0055, 0.0122,
         0.0156, 0.6561],
        [0.0130, 0.1354, 0.0100, 0.0267, 0.0626, 0.0075, 0.0537, 0.0054, 0.0121,
         0.0155, 0.6581],
        [0.0128, 0.1347, 0.0100, 0.0267, 0.0630, 0.0075, 0.0539, 0.0054, 0.0121,
         0.0155, 0.6584],
        [0.0129, 0.1336, 0.0100, 0.0269, 0.0631, 0.0076, 0.0544, 0.0054, 0.0123,
         0.0156, 0.6583],
        [0.0132, 0.1319, 0.0104, 0.0276, 0.0638, 0.0079, 0.0558, 0.0057, 0.0130,
         0.0161, 0.6546],
        [0.014

In [29]:
def greedy_decode(logits):
    # logits: (T, num_classes)
    indices = torch.argmax(logits, dim=1)  # (T,)
    
    # collapse consecutive repeats
    collapsed = []
    prev = None
    for idx in indices:
        idx = idx.item()
        if idx != prev:
            collapsed.append(idx)
        prev = idx
    
    # remove blanks (class 10)
    digits = [c for c in collapsed if c != 10]
    return "".join(str(d) for d in digits)

# run on a few val images
model.eval()
device = next(model.parameters()).device

for i in range(10):
    image, label, width = val_dataset[i]
    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(device))  # (T, 1, 11)
        indices = torch.argmax(logits[:, 0, :], dim=1)
        print(indices.tolist())
        pred = greedy_decode(logits[:, 0, :])
    print(f"pred: {pred} | true: {''.join(str(d) for d in label)}")

[1, 1, 10, 10, 10, 10, 10, 10, 10, 10, 6, 10, 10]
pred: 16 | true: 114
[1, 1, 1, 10, 10, 10, 10, 10, 10, 10, 10, 10]
pred: 1 | true: 3128
[1, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
pred: 1 | true: 3128
[9, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
pred: 9 | true: 1138
[6, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
pred: 6 | true: 28
[9, 10, 10, 10, 10, 10, 10, 10, 10, 10]
pred: 9 | true: 1138
[10, 10, 10, 10, 10, 10, 10, 10, 6, 10, 10]
pred: 6 | true: 1138
[9, 10, 10, 10, 10, 10, 10, 10, 10, 10]
pred: 9 | true: 1138
[9, 6, 10, 10, 10, 10, 10, 10, 10, 10, 10]
pred: 96 | true: 1138
[9, 9, 10, 10, 10, 10, 10, 10, 10, 10, 10]
pred: 9 | true: 3
